# Financial Fraud Detection with Blind Insight Data

This notebook demonstrates how to use financial fraud data from Blind Insight for machine learning tasks with scikit-learn.

## Prerequisites

1. Install required packages:
   ```bash
   pip install -r requirements.txt
   ```

2. Ensure the backend API is running:
   ```bash
   cd cube-server
   node index.js
   ```

3. Make sure you have uploaded the financial fraud dataset to Blind Insight using the importer tool.


## Setup and Imports


In [1]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Import the Blind Insight client
import sys
sys.path.append('.')
from blind_insight_client import BlindInsightClient, load_fraud_from_blind


/Users/ivan/code/blindinsight/importer/jupyter-integration/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


## Load Data from Blind Insight

Instead of using standard datasets, we load financial fraud data from Blind Insight.

**Note**: Update the organization, dataset_slug, and schema_slug to match your Blind Insight setup.


### Workflows
- **Encrypted-only (recommended)**: Use Blind Insight encrypted primitives (count, avg, min, max, ranges) and keep rows encrypted. See the *Encrypted Aggregation Classifier* section below.
- **Plaintext ML (illustrative only)**: Decrypt data and run scikit-learn directly. This is not encrypted and is kept for reference/testing.


In [2]:
# Configuration - Plaintext ML example (illustrative)
# NOTE: This decrypts data; for encrypted-only workflows see the section below.
ORGANIZATION = "demo"
DATASET_SLUG = "financial"
SCHEMA_SLUG = "antifraud"
API_URL = "https://proxy.local.blindinsight.io/"

# Plaintext load (decrypt=True inside load_fraud_from_blind)
# Load fraud detection data from Blind Insight
# The dataset contains fraud types: card_fraud, mule_account, unusual_activity, 
# account_takeover, synthetic_identity, suspicious_transfer
X, y = load_fraud_from_blind(
    organization=ORGANIZATION,
    dataset_slug=DATASET_SLUG,
    schema_slug=SCHEMA_SLUG,
    api_url=API_URL,
    decrypt=True,  # plaintext for scikit-learn; not encrypted
    feature_cols=["risk_level", "year", "month", "day"],  # Specify fraud features
    target_col="fraud_type"  # Use fraud_type as target (can also use "is_active" for binary)
)

print(f"Loaded {X.shape[0]} samples with {X.shape[1]} features")
print(f"Feature shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"Unique classes: {np.unique(y)}")
print(f"Class distribution: {np.bincount(y)}")
print(f"\nNote: Classes are encoded as integers. Original fraud types include:")
print(f"  card_fraud, mule_account, unusual_activity, account_takeover,")
print(f"  synthetic_identity, suspicious_transfer")


: 

## Prepare Data for Classification

For fraud detection, we'll use risk_level and year as the main features. We'll focus on binary classification 
by selecting the two most common fraud types, or use all classes for multi-class classification.


In [14]:
# For fraud detection, use risk_level and year as features
# Select a subset for binary classification (two most common fraud types)
unique_classes, class_counts = np.unique(y, return_counts=True)
# Get indices of the two most common classes
top_two_indices = np.argsort(class_counts)[-2:]
top_two_classes = unique_classes[top_two_indices]

# Filter data to include only the two most common fraud types
mask = np.isin(y, top_two_classes)
X_subset = X[mask]
y_subset = y[mask]

# Use risk_level and year as features (indices 0 and 1)
X_subset = X_subset[:, [0, 1]]  # risk_level, year

# Re-encode labels to 0 and 1 for binary classification
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y_subset = le.fit_transform(y_subset)

print(f"Subset shape: {X_subset.shape}")
print(f"Target classes in subset: {np.unique(y_subset)}")
print(f"Class distribution: {np.bincount(y_subset)}")
print(f"Features used: risk_level, year")


Subset shape: (100, 2)
Target classes in subset: [0 1]


In [15]:
print(f"Sample features (risk_level, year):\n{X_subset[:10]}")
print(f"\nSample targets:\n{y_subset[:10]}")
print(f"\nFeature statistics:")
print(f"Risk level - Min: {X_subset[:, 0].min()}, Max: {X_subset[:, 0].max()}, Mean: {X_subset[:, 0].mean():.2f}")
print(f"Year - Min: {X_subset[:, 1].min()}, Max: {X_subset[:, 1].max()}, Mean: {X_subset[:, 1].mean():.2f}")


[[51 35]
 [49 30]
 [47 32]
 [46 31]
 [50 36]
 [54 39]
 [46 34]
 [50 34]
 [44 29]
 [49 31]
 [54 37]
 [48 34]
 [48 30]
 [43 30]
 [58 40]
 [57 44]
 [54 39]
 [51 35]
 [57 38]
 [51 38]
 [54 34]
 [51 37]
 [46 36]
 [51 33]
 [48 34]
 [50 30]
 [50 34]
 [52 35]
 [52 34]
 [47 32]
 [48 31]
 [54 34]
 [52 41]
 [55 42]
 [49 31]
 [50 32]
 [55 35]
 [49 36]
 [44 30]
 [51 34]
 [50 35]
 [45 23]
 [44 32]
 [50 35]
 [51 38]
 [48 30]
 [51 38]
 [46 32]
 [53 37]
 [50 33]
 [70 32]
 [64 32]
 [69 31]
 [55 23]
 [65 28]
 [57 28]
 [63 33]
 [49 24]
 [66 29]
 [52 27]
 [50 20]
 [59 30]
 [60 22]
 [61 29]
 [56 29]
 [67 31]
 [56 30]
 [58 27]
 [62 22]
 [56 25]
 [59 32]
 [61 28]
 [63 25]
 [61 28]
 [64 29]
 [66 30]
 [68 28]
 [67 30]
 [60 29]
 [57 26]
 [55 24]
 [55 24]
 [58 27]
 [60 27]
 [54 30]
 [60 34]
 [67 31]
 [63 23]
 [56 30]
 [55 25]
 [55 26]
 [61 30]
 [58 26]
 [50 23]
 [56 27]
 [57 30]
 [57 29]
 [62 29]
 [51 25]
 [57 28]]
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0

## Train Logistic Regression Model


In [16]:
# Create an instance of Logistic Regression Classifier and fit the data
logreg = LogisticRegression(C=1e5, max_iter=1000)
clf = logreg.fit(X_subset, y_subset)

print("Model trained successfully!")
print(clf)


Model trained successfully!
LogisticRegression(C=100000.0, max_iter=1000)


## Evaluate Model Accuracy


In [17]:
# Make predictions
y_pred = clf.predict(X_subset)

# Calculate accuracy
accuracy = accuracy_score(y_subset, y_pred)
print(f"Accuracy: {accuracy:.4f} ({accuracy * 100:.2f}%)")


Accuracy: 1.0000 (100.00%)


Blind 

In [18]:
# Example: Use encrypted filters to narrow dataset before decryption
# Encrypted aggregation-based classifier (no decryption)
# Two-class demo: setosa vs versicolor using petal-length feature
from blind_insight_client import BlindInsightClient

client = BlindInsightClient(api_url=API_URL)
feature = "petal-length"
class_a = "I. setosa"
class_b = "I. versicolor"

# Helper to extract aggregation value (supports both shapes)
# Shape A: {records: [{data: {value: X}}]}
# Shape B: {records: [{value: X, aggregation_type: ...}]}

def agg_value(resp):
    recs = resp.get("records", [])
    if not recs:
        raise ValueError(f"Unexpected aggregation response: {resp}")
    rec0 = recs[0]
    # Shape A
    if "data" in rec0 and isinstance(rec0["data"], dict) and "value" in rec0["data"]:
        v = rec0["data"].get("value")
        return float(v) if v is not None else 0.0
    # Shape B
    if "value" in rec0:
        v = rec0.get("value")
        return float(v) if v is not None else 0.0
    raise ValueError(f"Unexpected aggregation response shape: {resp}")

# NOTE: Blind Insight aggregation supports counts on encrypted numbers; to avoid
# unsupported floating-point avg on some deployments, we do a threshold search
# using encrypted counts only (no decryption).

# Candidate thresholds over petal-length range (Iris ~0 to ~7)
thresholds = np.linspace(0.0, 7.0, 71)  # step 0.1

# Precompute totals per class
count_a_total = agg_value(
    client.aggregate(
        organization=ORGANIZATION,
        dataset_slug=DATASET_SLUG,
        schema_slug=SCHEMA_SLUG,
        agg_filter=f"{feature}:count(0~1000)",
        extra_filters=[f"species:{class_a}"],
        decrypt=False
    )
)
count_b_total = agg_value(
    client.aggregate(
        organization=ORGANIZATION,
        dataset_slug=DATASET_SLUG,
        schema_slug=SCHEMA_SLUG,
        agg_filter=f"{feature}:count(0~1000)",
        extra_filters=[f"species:{class_b}"],
        decrypt=False
    )
)

# Thresholds over integer-scaled petal-length (e.g., 0..700 if 0..7 *10)
# Narrow search window and coarser step to speed up
thresholds = range(0, 201, 20)   # was 0..700 step 10

best_acc = -1
best_t = None
best_counts = None

for t in thresholds:
    count_a_left = agg_value(
        client.aggregate(
            organization=ORGANIZATION,
            dataset_slug=DATASET_SLUG,
            schema_slug=SCHEMA_SLUG,
            agg_filter=f"{feature}:count(<{t})",
            extra_filters=[f"species:{class_a}"],
            decrypt=False
        )
    )
    count_b_left = agg_value(
        client.aggregate(
            organization=ORGANIZATION,
            dataset_slug=DATASET_SLUG,
            schema_slug=SCHEMA_SLUG,
            agg_filter=f"{feature}:count(<{t})",
            extra_filters=[f"species:{class_b}"],
            decrypt=False
        )
    )

    correct = count_a_left + (count_b_total - count_b_left)
    total = count_a_total + count_b_total
    acc = correct / total if total else 0.0

    if acc > best_acc:
        best_acc = acc
        best_t = t
        best_counts = (count_a_left, count_b_left, correct, total)

print(f"Best threshold on encrypted counts: {best_t:.3f}")
print(f"Encrypted-rule accuracy: {best_acc:.4f} ({best_acc*100:.2f}%)")
print(f"Counts at best threshold -> class_a_left: {best_counts[0]:.0f}, class_b_left: {best_counts[1]:.0f}, correct: {best_counts[2]:.0f}/{best_counts[3]:.0f}")

print("\nRule: predict setosa if petal-length < threshold, else versicolor (encrypted counts only)")


: 

## Alternative: Using the Client Directly

You can also use the client directly for more control over the data loading process.


## Encrypted Aggregation Classifier (no decryption)

This section demonstrates classification using only Blind Insight's encrypted primitives:
- Use encrypted filters and aggregations (count/min/max/ranges) on encrypted data
- Never decrypt rows
- Build a simple rule-based classifier using aggregate stats only

**Dataset note**: For this demo we assume the numeric features are integer-scaled (e.g., original floats multiplied by 10) so that counts/ranges operate on integers. Adjust ranges accordingly if you change the scaling.

Approach (two-class demo: setosa vs versicolor):
1. Search thresholds over integer-scaled petal-length using encrypted `count(<t)` per class.
2. Pick the threshold with best accuracy on encrypted counts.
3. No row-level decryption; only aggregates are returned.


## Encrypted averages: sepal width per class
Use Blind Insight encrypted `avg` to compute the mean sepal width for each class (integer-scaled).


In [4]:
# Encrypted avg of sepal-width per class (integer-scaled data)
from blind_insight_client import BlindInsightClient

client = BlindInsightClient(api_url=API_URL)
feature = "sepal-width"
classes = ["I. setosa", "I. versicolor", "I. virginica"]

# Reuse agg_value from earlier cell if in scope; redefine defensively

def agg_value(resp):
    recs = resp.get("records", [])
    if not recs:
        raise ValueError(f"Unexpected aggregation response: {resp}")
    rec0 = recs[0]
    if "data" in rec0 and isinstance(rec0["data"], dict) and "value" in rec0["data"]:
        v = rec0["data"].get("value")
        return float(v) if v is not None else 0.0
    if "value" in rec0:
        v = rec0.get("value")
        return float(v) if v is not None else 0.0
    raise ValueError(f"Unexpected aggregation response shape: {resp}")

for cls in classes:
    resp = client.aggregate(
        organization=ORGANIZATION,
        dataset_slug=DATASET_SLUG,
        schema_slug=SCHEMA_SLUG,
        agg_filter=f"{feature}:avg(0~1000)",
        extra_filters=[f"species:{cls}"],
        decrypt=False  # keep encrypted; avg is computed inside Blind Insight
    )
    mean_val = agg_value(resp)
    print(f"Mean {feature} for {cls}: {mean_val:.3f}")



: 

## Encrypted averages per batch of 10 rows (all features)
Compute integer-scaled means for each feature in batches of 10 rows using encrypted `avg` with a range filter on `dataset-order`.


In [5]:
# Batch means (integer-scaled) over all rows, in batches of 10
from blind_insight_client import BlindInsightClient

client = BlindInsightClient(api_url=API_URL)
features = ["sepal-length", "sepal-width", "petal-length", "petal-width"]  # integer-scaled
batch_size = 10

# Reuse agg_value if defined; otherwise define here

def agg_value(resp):
    recs = resp.get("records", [])
    if not recs:
        raise ValueError(f"Unexpected aggregation response: {resp}")
    rec0 = recs[0]
    if "data" in rec0 and isinstance(rec0["data"], dict) and "value" in rec0["data"]:
        v = rec0["data"].get("value")
        return float(v) if v is not None else 0.0
    if "value" in rec0:
        v = rec0.get("value")
        return float(v) if v is not None else 0.0
    raise ValueError(f"Unexpected aggregation response shape: {resp}")

# Determine total rows via encrypted count on dataset-order
TOTAL_MAX = 100000  # a large upper bound
count_resp = client.aggregate(
    organization=ORGANIZATION,
    dataset_slug=DATASET_SLUG,
    schema_slug=SCHEMA_SLUG,
    agg_filter=f"dataset-order:count(0~{TOTAL_MAX})",
    decrypt=False
)
total_rows = int(agg_value(count_resp))
print(f"Total rows detected: {total_rows}")

start = 0
while start < total_rows:
    end = min(start + batch_size - 1, total_rows - 1)
    batch_filter = f"dataset-order:{start}~{end}"
    print(f"\nBatch {start}-{end}:")
    for feature in features:
        resp = client.aggregate(
            organization=ORGANIZATION,
            dataset_slug=DATASET_SLUG,
            schema_slug=SCHEMA_SLUG,
            agg_filter=f"{feature}:avg(0~1000)",
            extra_filters=[batch_filter],
            decrypt=False
        )
        mean_val = agg_value(resp)
        print(f"  mean {feature}: {mean_val:.3f}")
    start += batch_size



: 

In [3]:
# Example: Load data as a pandas DataFrame for exploration
client = BlindInsightClient(api_url=API_URL)

# Check API health
health = client.health_check()
print(f"API Status: {health}")

# Load full dataset as DataFrame
df = client.load_data(
    organization=ORGANIZATION,
    dataset_slug=DATASET_SLUG,
    schema_slug=SCHEMA_SLUG,
    limit=150
)

print(f"\nDataFrame shape: {df.shape}")
print(f"\nColumn names: {df.columns.tolist()}")
print(f"\nFirst few rows:")
print(df.head())
print(f"\nDataFrame info:")
print(df.info())


: 

## Summary

This notebook demonstrates:

1. ✅ Loading data from Blind Insight instead of sklearn datasets
2. ✅ Using the data with scikit-learn for machine learning
3. ✅ Training a logistic regression classifier
4. ✅ Evaluating model accuracy

The key difference from the standard scikit-learn example is that instead of:
```python
iris = datasets.load_iris()
```

We use:
```python
X, y = load_iris_from_blind(organization, dataset_slug, schema_slug)
```

This allows data scientists to work with encrypted, privacy-preserving data stored in Blind Insight while using standard ML libraries and workflows.
